In [8]:
import os
import json
import warnings
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from mp_api.client import MPRester
from pymatgen.core import Element
warnings.filterwarnings("ignore")

load_dotenv()
API_KEY = os.getenv("MP_API_KEY")
print(f"API key loaded: {API_KEY is not None}")

API key loaded: True


In [9]:
LI_FIELDS = [
    "material_id",
    "formula_pretty",
    "structure",
    "band_gap",
    "energy_above_hull",
    "formation_energy_per_atom",
    "volume",
    "density",
    "nsites",
    'nelements',
    "elements",
    "symmetry",
]

In [10]:
with MPRester(API_KEY) as mpr:
    li_results = mpr.materials.summary.search(
        elements=["Li"],
        band_gap=(1.0, None),
        energy_above_hull=(0.0, 0.1),
        fields = LI_FIELDS
    )
    print(f"Li-containing materials retrieved: {len(li_results)}")

with MPRester(API_KEY) as mpr:
    na_results = mpr.materials.summary.search(
        elements=["Na"],
        band_gap=(1.0, None),
        energy_above_hull=(0.0, 0.1),
        fields = LI_FIELDS
    )
    print(f"Na-containing materials retrieved: {len(na_results)}")

Retrieving SummaryDoc documents: 100%|██████████| 8779/8779 [00:08<00:00, 1025.86it/s]


Li-containing materials retrieved: 8779


Retrieving SummaryDoc documents: 100%|██████████| 6873/6873 [00:13<00:00, 505.13it/s]

Na-containing materials retrieved: 6873


In [11]:
all_results = li_results + na_results
seen_ids = set()
unique_results = []
for entry in all_results:
    if entry["material_id"] not in seen_ids:
        unique_results.append(entry)
        seen_ids.add(entry["material_id"])
        
print(f"Unique materials after deduplication: {len(unique_results)}")


Unique materials after deduplication: 15362


In [12]:
def extract_features(entry):
    """Extract compositional and structural features from a Materials Project entry"""
    try:
        structure = entry.structure
        elements = [str(el) for el in structure.composition.elements]

        # Mobile ion features
        li_frac = structure.composition.get_atomic_fraction("Li") if "Li" in elements else 0.0
        na_frac = structure.composition.get_atomic_fraction("Na") if "Na" in elements else 0.0

        # Mean atomic features across all sites
        atomic_numbers = [site.specie.Z for site in structure]
        electronegativities = []
        ionic_radii = []
        atomic_radii = []

        for site in structure:
            el = site.specie
            electronegativities.append(el.X if el.X else np.nan)
            atomic_radii.append(el.atomic_radius if el.atomic_radius else np.nan)
            # Use common oxidation state ionic radius where available
            try:
                ox = el.common_oxidation_states[0] if el.common_oxidation_states else None
                if ox is not None and ox in el.ionic_radii:
                    ionic_radii.append(float(el.ionic_radii[ox]))
                else:
                    ionic_radii.append(np.nan)
            except:
                ionic_radii.append(np.nan)
        return {
            "material_id": entry.material_id,
            "formula": entry.formula_pretty,
            "band_gap": entry.band_gap,
            "energy_above_hull": entry.energy_above_hull,
            "formation_energy_per_atom": entry.formation_energy_per_atom,
            "volume": entry.volume,
            "density": entry.density,
            "nsites": entry.nsites,
            "nelements": entry.nelements,
            "spacegroup": entry.symmetry.symbol if entry.symmetry else None,
            "crystal_system": entry.symmetry.crystal_system if entry.symmetry else None,
            "li_fraction": li_frac,
            "na_fraction": na_frac,
            "mean_atomic_number": np.nanmean(atomic_numbers),
            "mean_electronegativity": np.nanmean(electronegativities),
            "std_electronegativity": np.nanstd(electronegativities),
            "mean_atomic_radius": np.nanmean(atomic_radii),
            "std_atomic_radius": np.nanstd(atomic_radii),
            "mean_ionic_radius": np.nanmean(ionic_radii),
            "std_ionic_radius": np.nanstd(ionic_radii),
        }
    except Exception as e:
        return None
    
rows = []
for i,entry in enumerate(unique_results):
    row = extract_features(entry)
    if row:
        rows.append(row)

df = pd.DataFrame(rows)
print(f"Final dataset shape: {df.shape}")


Final dataset shape: (15362, 20)


In [13]:
structures = {}
for entry in unique_results:
    try:
        structures[entry.material_id] = entry.structure.as_dict()
    except Exception as e:
        print(f"Failed to extract structure for {entry.material_id}: {e}")

os.makedirs("../data/raw", exist_ok=True)
with open("../data/raw/structures.json", "w") as f:
    json.dump(structures, f)
print(f"Saved {len(structures)} structures to /data/raw/structures.json")

Saved 15362 structures to /data/raw/structures.json


In [14]:
df.to_csv("../data/raw/electrolyte_candidates.csv", index=False)
print(f"Saved raw dataset: {df.shape[0]} materials, {df.shape[1]} features")

Saved raw dataset: 15362 materials, 20 features


In [15]:
print("=== Data Quality Report ===")
print(f"Total materials: {len(df)}")
print(f"Li-containing: {(df['li_fraction'] > 0).sum()}")
print(f"Na-containing: {(df['na_fraction'] > 0).sum()}")
print(f"Both Li and Na: {((df['li_fraction'] > 0) & (df['na_fraction'] > 0)).sum()}")
print(f"\nBand gap range: {df['band_gap'].min():.2f} — {df['band_gap'].max():.2f} eV")
print(f"Mean band gap: {df['band_gap'].mean():.2f} eV")
print(f"\nMissing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nCrystal systems:")
print(df['crystal_system'].value_counts())

=== Data Quality Report ===
Total materials: 15362
Li-containing: 8779
Na-containing: 6873
Both Li and Na: 290

Band gap range: 1.00 — 8.72 eV
Mean band gap: 2.84 eV

Missing values per column:
Series([], dtype: int64)

Crystal systems:
crystal_system
Monoclinic      5100
Triclinic       5099
Orthorhombic    2367
Trigonal        1004
Tetragonal       797
Cubic            735
Hexagonal        260
Name: count, dtype: int64


In [16]:
df_clean = df.dropna(thresh=len(df.columns)-3)
df_clean.to_csv("../data/processed/solid_electrolyte_features.csv", index=False)
print(f"\nProcessed dataset saved: {df_clean.shape[0]} materials, {df_clean.shape[1]} features (after cleaning)")


Processed dataset saved: 15362 materials, 20 features (after cleaning)
